In [ ]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler()
    ]
)

examples = [
    {
        "movie": "탑건",
        "output": "🛩️👨‍✈️🔥"
    },
    {  
        "movie": "대부",
        "output": "👨‍👨‍👦🔫🍝"
    }
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{movie}"),
        ("ai", "{output}")
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
    memory_key="chat_history"
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "영화를 찾아서, 예제와 같이 3개의 이모티콘으로 표현해줘"),
        few_shot_prompt,
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{movie}")
    ]
)

def load_memory(_):
    return memory.load_memory_variables({})["chat_history"]

chain = RunnablePassthrough.assign(chat_history=load_memory) | final_prompt | llm

def invoke_chain(input):
    result = chain.invoke({"movie": input})
    memory.save_context(
        {"movie": input},
        {"output": result.content}
    )

invoke_chain("주토피아")




🐰🦊🚔